# Quantization Aware Training

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 2s 0us/step


## Load Baseline Model for MNIST (CNN)

In [ ]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [ ]:
_, baseline_model_accuracy = model.evaluate(test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Convert to TFLite model (Baseline model)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_baseline_model = converter.convert()

In [ ]:
litert_model_path = "/content/drive/MyDrive/files/save"

In [ ]:
import pathlib

tflite_models_dir = pathlib.Path(litert_model_path)
tflite_models_dir.mkdir(exist_ok=True, parents=True)

tflite_baseline_model_file = tflite_models_dir/"mnist_baseline_model.tflite"
size_baseline_model = tflite_baseline_model_file.write_bytes(tflite_baseline_model)

In [ ]:
interpreter_base = tf.lite.Interpreter(model_content=tflite_baseline_model)
interpreter_base.allocate_tensors()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Define Model with Quantization Aware Training


In [ ]:
q_aware_model = tfmot.quantization.keras.quantize_model(model)

q_aware_model.compile(optimizer='adam',
                      loss=keras.losses.SparseCategoricalCrossentropy(),
                      metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28)            3         
 yer)                                                            
                                                                 
 quant_reshape (QuantizeWra  (None, 28, 28, 1)         1         
 pperV2)                                                         
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        387       
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        4

## Fine-tune for QAT (training data 의 subset을 이용)

In [ ]:
train_images_subset = train_images
train_labels_subset = train_labels

q_aware_model.fit(train_images_subset, train_labels_subset, batch_size=500, epochs=1, validation_split=0.1)

108/108 [==============================] - 45s 386ms/step - loss: 0.0107 - accuracy: 0.9973 - val_loss: 0.0279 - val_accuracy: 0.9933


## LiteRT 모델로 전환 (실제 weight의 양자화 적용, Static Quantization)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_qat_model = converter.convert()

tflite_qat_model_file = tflite_models_dir/"mnist_qat_model.tflite"
size_qat_model = tflite_qat_model_file.write_bytes(tflite_qat_model)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
print("Size of Baseline LiteRT Model file : {}".format(size_baseline_model))
print("Size of QAT LiteRT Model file : {}".format(size_qat_model))

Size of Baseline LiteRT Model file : 234392
Size of QAT LiteRT Model file : 64904


In [ ]:
interpreter_qat = tf.lite.Interpreter(model_content=tflite_qat_model)
interpreter_qat.allocate_tensors()

input_dtype = interpreter_qat.get_input_details()[0]['dtype']
output_dtype = interpreter_qat.get_output_details()[0]['dtype']

print("input dtype : {}".format(input_dtype))
print("output dtype : {}".format(output_dtype))

input dtype : <class 'numpy.float32'>
output dtype : <class 'numpy.float32'>


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
test_input = np.expand_dims(test_images[0],axis=0)

interpreter_qat.set_tensor(interpreter_qat.get_input_details()[0]['index'], test_input)
interpreter_qat.invoke()

prediction = interpreter_qat.get_tensor(interpreter_qat.get_output_details()[0]['index'])

print(prediction)
print(np.argmax(prediction))
print(test_labels[0])

[[0.         0.         0.         0.         0.         0.
  0.         0.99609375 0.         0.        ]]
7
7


In [ ]:
def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  prediction_digits = []
  for i, test_image in enumerate(test_images):
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
interpreter_qat = tf.lite.Interpreter(model_content=tflite_qat_model)
interpreter_qat.allocate_tensors()

print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_qat))

0.99
0.9926


## model graph 구조 확인

In [ ]:
model_path_str = '/content/drive/MyDrive/files/save/mnist_qat_model.tflite'

In [ ]:
tf.lite.experimental.Analyzer.analyze(model_path=model_path_str)

=== /content/drive/MyDrive/files/save/mnist_qat_model.tflite ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the QUANTIZE op takes
tensor #0 as input and produces tensor #10 as output.

Subgraph#0 main(T#0) -> [T#27]
  Op#0 QUANTIZE(T#0) -> [T#10]
  Op#1 SHAPE(T#10) -> [T#11]
  Op#2 STRIDED_SLICE(T#11, T#2[0], T#1[1], T#1[1]) -> [T#12]
  Op#3 PACK(T#12, T#4[28], T#4[28], T#3[1]) -> [T#13]
  Op#4 RESHAPE(T#10, T#13) -> [T#14]
  Op#5 CONV_2D(T#14, T#15, T#7[-24795, -5708, -8377, -1257, -16268, ...]) -> [T#16]
  Op#6 MAX_POOL_2D(T#16) -> [T#17]
  Op#7 CONV_2D(T#17, T#18, T#6[-311, -470, -337, 126, 211, ...]) -> [T#19]
  Op#8 MAX_POOL_2D(T#19) -> [T#20]
  Op#9 RESHAPE(T#20, T#5[-1, 400]) -> [T#21]
  Op#10 FULLY_CONNECTED(T#21, T#22, T#8[45, -126, -136, 45, -53, ...]) -> [T#23]
  Op#11 FULLY_CONNECTED(T#23, T#24, T#9[-133, 58, -115, 56, -182, ...]) -> [T#25]
  Op#12 SOFTMAX(T#25) -> [T#26]
  Op#13 D